# Orbital PPO basic

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import ray
import json
import numba as nb
import numpy as np
import matplotlib.pyplot as plt
import pickle
from itertools import product
from pathlib import Path

from tqdm.auto import trange, tqdm
from math import radians, pi, sin
from numba.experimental import jitclass
from IPython.display import display, JSON
import ipywidgets as widgets

from ray import tune
from ray.tune.registry import register_env
from ray.rllib.agents import ppo
from ray.rllib.models import ModelCatalog
from ray.rllib.models.tf.tf_modelv2 import TFModelV2
from ray.rllib.models.tf.fcnet import FullyConnectedNetwork
from ray.tune.utils.log import Verbosity
from ray.tune import JupyterNotebookReporter
# Verbosity.V0_MINIMAL
# Verbosity.V1_EXPERIMENT
# Verbosity.V2_TRIAL_NORM
# Verbosity.V3_TRIAL_DETAILS

from cw.filters import smooth_signal
from cw.vdom import hyr
from traj1.logger import Logger
from environment import LauncherV1Orbital, Stage, AP_PITCH_RATE_CONTROL, wrap_angle
from sufcnet import SUFullyConnectedNetwork

## Configure

In [3]:
# tensorboard --logdir=~/ray_results --port=8270 --host=0.0.0.0
hyr(ray.init(dashboard_host="0.0.0.0", num_gpus=1))

2022-10-28 05:08:44,411	INFO worker.py:1509 -- Started a local Ray instance. View the dashboard at 192.168.0.107:8265 
/home/deck/miniconda3/envs/traj1/lib/python3.10/site-packages/ray/_private/worker.py:976: UserWarning: len(ctx) is deprecated. Use len(ctx.address_info) instead.
  warnings.warn("len(ctx) is deprecated. Use len(ctx.address_info) instead.")


Python version:,3.10.6
Ray version:,2.0.0
Dashboard:,http://192.168.0.107:8265


In [4]:
register_env("LauncherV1Orbital", LauncherV1Orbital)
ModelCatalog.register_custom_model("SUFullyConnectedNetwork", SUFullyConnectedNetwork)

In [5]:
init_dir = Path('./init_tables')
with (init_dir / "2022_10_21__batch_1.json").open("r") as f:
    selected_initial_conditions = json.load(f)

## Learn

In [6]:
tune.run(
    ppo.PPOTrainer,
    config={
        "gamma": 0.995,
        "vf_clip_param": 1.,
        "clip_param": 0.2,
        
        "env": "LauncherV1Orbital",
        "num_workers": 4,
        "num_gpus": 0,
        "batch_mode": "complete_episodes",
        "env_config": {
            "dt": 0.05,
            "init_table": selected_initial_conditions['1']
        },
        "model": {
            "custom_model": "SUFullyConnectedNetwork",
            "custom_model_config": {
                "dropouts": [0.1, 0.1, 0.1],
                "training": True},
            "fcnet_hiddens": [64, 64, 64],
            "fcnet_activation": "relu",
        },
    },
    checkpoint_freq=1,
    checkpoint_at_end=True,
    local_dir="./rl_results",
    name="ppo_basic",
    progress_reporter=JupyterNotebookReporter(overwrite=True),
    verbose=0,
#     resume=True,
#     restore="~/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_8e488_00000_0_2021-05-15_01-19-12/checkpoint_001230",
)

(PPO pid=179924) 2022-10-28 05:08:53,781	INFO algorithm.py:1871 -- Your framework setting is 'tf', meaning you are using static-graph mode. Set framework='tf2' to enable eager execution with tf2.x. You may also then want to set eager_tracing=True in order to reach similar execution speed as with static-graph mode.
(PPO pid=179924) 2022-10-28 05:08:53,781	INFO ppo.py:378 -- In multi-agent mode, policies will be optimized sequentially by the multi-GPU optimizer. Consider setting simple_optimizer=True if this doesn't work for you.
(PPO pid=179924) 2022-10-28 05:08:53,784	INFO algorithm.py:351 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
(RolloutWorker pid=179972) /home/deck/repos/thesis/code/trajectory_optimization_1/traj1/environments/launcher_v1/simulation.py:121: NumbaWarning: Cannot cache compiled function "new_simulation" as it uses dynamic globals (such as ctypes pointers and large global arrays)
(RolloutWorker pi

TuneError: ('Trials did not complete', [PPO_LauncherV1Orbital_d6230_00000])